In [11]:
import torch
import torch.nn as nn
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision import transforms
from torch.utils.data import Dataset
from torchmetrics.classification import BinaryAccuracy
from torchmetrics.classification import MulticlassAccuracy
from torch.optim import lr_scheduler
import torch.nn.functional as F

import os
import numpy as np
import random
from PIL import Image
import copy

from tqdm.auto import tqdm



#######################
import multiprocessing
multiprocessing.set_start_method('spawn', force=True)


In [12]:
class SegDataset(Dataset):

    def __init__(self, image_dir, mask_dir, img_transform = None, mask_transform = None):

        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_transform = img_transform
        self.mask_transform = mask_transform

        self.images = sorted(os.listdir(image_dir))
        self.masks = sorted(os.listdir(mask_dir))

    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, index):
        image = Image.open(os.path.join(self.image_dir, self.images[index]))
        mask = Image.open(os.path.join(self.mask_dir, self.masks[index]))

        if self.img_transform is not None and self.mask_transform is not None:

            # Capture a random seed
            seed = np.random.randint(2147483647) 
            
            # Apply to image
            random.seed(seed) 
            torch.manual_seed(seed) 
            image = self.img_transform(image)

            # Apply to mask using the SAME seed
            random.seed(seed) 
            torch.manual_seed(seed) 
            mask = self.mask_transform(mask)


        return image, mask


In [13]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        
        self.in_channels = in_channels
        self.out_channels = out_channels

        self.block = nn.Sequential(
            nn.Conv2d(in_channels = self.in_channels, 
                      out_channels = self.out_channels,
                      kernel_size = 3, 
                      stride = 1,
                      padding = 1,
                      bias = False),
            nn.BatchNorm2d(self.out_channels),
            nn.ReLU(inplace = True),


            nn.Conv2d(in_channels = self.out_channels, 
                      out_channels = self.out_channels,
                      kernel_size = 3, 
                      stride = 1,
                      padding = 1,
                      bias = False),
            nn.BatchNorm2d(self.out_channels),
            nn.ReLU(inplace = True)
        )


    def forward(self, x):

        x = self.block(x)

        return x
        

In [14]:
class UNet(nn.Module):
    def __init__(self, in_channels = 3, num_classes = 1, feature_list = [32, 64, 128, 256, 512], bottleneck_size = 1024):
        super(UNet, self).__init__()

        self.num_classes = num_classes
        self.feature_list = feature_list
        self.bottleneck_size = bottleneck_size


        # Model Architecture
        self.encoder = nn.ModuleList()
        self.bottleneck = ConvBlock(
            in_channels = self.feature_list[-1],
            out_channels = self.bottleneck_size
        )
        self.decoder = nn.ModuleList()
        self.classifier = nn.Conv2d(
            in_channels = feature_list[0],
            out_channels = self.num_classes,
            kernel_size = 1
        )

        self.pool = nn.MaxPool2d(kernel_size = 2, stride = 2)

        self._make_encoder(in_channels, self.feature_list)
        self._make_decoder(self.feature_list)
        
        


    def _make_encoder(self, in_channels, feature_list):

        ch = in_channels

        for feature in self.feature_list:

            encoder_level = ConvBlock(
                in_channels = ch,
                out_channels = feature
            )

            self.encoder.append(encoder_level)
            self.encoder.append(self.pool)

            ch = feature

    
    def _make_decoder(self, feature_list):

        for feature in reversed(self.feature_list):

            upconv = nn.ConvTranspose2d(
                in_channels = feature * 2,
                out_channels = feature,
                kernel_size = 2,
                stride = 2
            )

            decoder_level = ConvBlock(
                in_channels = feature * 2,
                out_channels = feature
            )

            self.decoder.append(upconv)
            self.decoder.append(decoder_level)


    def forward(self, x):
        
        skips = []

        for i in range(0,  len(self.encoder), 2):       #   Iterate over Encoder Module List, starting from index 0, with a step size of 2 to skip over previous maxpool layer

            x = self.encoder[i](x)      #   Run ConvBlock

            skips.append(x)             #   Save output for skip connection

            x = self.encoder[i + 1](x)  #   Run MaxPool layer  

        x = self.bottleneck(x)

        skips = skips[::-1]   #   Reverse skip connections to start from the last one

        for i in range(0, len(self.decoder), 2):
            x = self.decoder[i](x)

            skip_connection = skips[i//2]   #   Step size of 2 skips over previous upconv layer, this indexing format selects the correct skip conenction

            x = torch.cat((skip_connection, x), dim = 1)

            x = self.decoder[i + 1](x)

        x = self.classifier(x)

        return x



In [15]:
def create_dataloaders(dataset_dir, batch_size, img_transform, mask_transform, val_img_transform, val_mask_transform):

    train_img_dir = os.path.join(dataset_dir, "train/images")
    train_mask_dir = os.path.join(dataset_dir, "train/masks")
    test_img_dir = os.path.join(dataset_dir, "test/images")
    test_mask_dir = os.path.join(dataset_dir, "test/masks")
    val_img_dir = os.path.join(dataset_dir, "val/images")
    val_mask_dir = os.path.join(dataset_dir, "val/masks")

    train_dataset = SegDataset(
        image_dir = train_img_dir, 
        mask_dir = train_mask_dir, 
        img_transform = img_transform,
        mask_transform = mask_transform
    )

    test_dataset = SegDataset(
        image_dir = test_img_dir,
        mask_dir = test_mask_dir,
        img_transform = val_img_transform,  
        mask_transform = val_mask_transform
    )

    val_dataset = SegDataset(
        image_dir = val_img_dir,
        mask_dir = val_mask_dir,
        img_transform = val_img_transform,  
        mask_transform = val_mask_transform
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=0, pin_memory=True)

    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                            num_workers=0, pin_memory=True)

    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                            num_workers=0, pin_memory=True)

    return train_loader, test_loader, val_loader

In [16]:
def mask_to_binary(x):
    return (x > 0).float()

def mask_to_long(x):
    return torch.from_numpy(np.array(x)).long().unsqueeze(0)

In [17]:
class HybridLoss(nn.Module):

    def __init__(self, num_classes, alpha = 0.5, weights = None, smooth = 1e-6):
        super(HybridLoss, self).__init__()

        self.num_classes = num_classes
        self.alpha = alpha
        self.smooth = smooth

        # Check if weights are provided and convert to tensor if necessary
        if weights is not None and not isinstance(weights, torch.Tensor):
            self.weights = torch.tensor(self.weights, dtype = torch.float32)

        else: 
            self.weights = weights

        # Initialize Pixel Loss
        if num_classes == 1:
            # pos_weight handles the imbalance of the positive class
            self.pixel_loss = nn.BCEWithLogitsLoss(pos_weight=self.weights)
        else:
            # weight handles the imbalance across all C classes
            self.pixel_loss = nn.CrossEntropyLoss(weight=self.weights)

    
    def forward(self, inputs, targets):
        if self.weights is not None:
            self.weights = self.weights.to(inputs.device)

        # Pixel Loss
        if self.num_classes == 1:
            pixel_loss = self.pixel_loss(inputs, targets.float())
            probs = torch.sigmoid(inputs)
            true_dist = targets.float()
        
        else:
            pixel_loss = self.pixel_loss(inputs, targets.long())
            probs = F.softmax(inputs, dim = 1)
            true_dist = F.one_hot(targets.squeeze(1).long(), num_classes = self.num_classes)
            true_dist = true_dist.permute(0, 3, 1, 2).float()


        # Dice Loss
        inputs_flat = probs.view(probs.size(0), probs.size(1), -1)
        true_flat = true_dist.view(true_dist.size(0), true_dist.size(1), -1)

        intersection = torch.sum(inputs_flat * true_flat, dim = 2)
        cardinality = torch.sum(inputs_flat + true_flat, dim = 2)

        dice_score =  (2. * intersection + self.smooth) / (cardinality + self.smooth)
        dice_loss_per_class = 1 - dice_score

        # Apply weights to Dice Loss if provided
        if self.num_classes > 1 and self.weights is not None:
            dice_loss = (dice_loss_per_class * self.weights).mean()
        
        else:
            dice_loss = dice_loss_per_class.mean()


        return (self.alpha * pixel_loss) + ((1 - self.alpha) * dice_loss)



In [18]:
def get_automated_weights(mask_dir, num_classes, cache_path="weights_cache.pt"):
    
    # Return cached weights if they exist
    if os.path.exists(cache_path):
        print(f"Loading cached weights from {cache_path}...")
        return torch.load(cache_path)
    
    print(f"Calculating automated weights from: {mask_dir}...")
    
    if num_classes == 1:
        counts = np.zeros(2)
    else:
        counts = np.zeros(num_classes)

    mask_files = [f for f in os.listdir(mask_dir) if not f.startswith('.')]

    for f in mask_files:
        mask = np.array(Image.open(os.path.join(mask_dir, f)))
        if mask.max() > 1:
            mask = (mask > 0).astype(np.uint8)
        unique, pixel_counts = np.unique(mask, return_counts=True)
        for val, count in zip(unique, pixel_counts):
            if val < len(counts):
                counts[int(val)] += count

    if num_classes == 1:
        weight = counts[0] / (counts[1] + 1e-6)
        weights = torch.tensor([weight])
    else:
        total_pixels = np.sum(counts)
        weights = total_pixels / (num_classes * counts + 1e-6)
        weights = weights / np.min(weights)
        weights = torch.tensor(weights, dtype=torch.float32)

    # Save for next run
    torch.save(weights, cache_path)
    print(f"Weights calculated and cached: {weights}")
    return weights

In [19]:
def Train_UNet(dataset_dir, num_epochs, learning_rate, num_classes, feature_list, bottleneck_size, image_size, batch_size):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Create directory to save model
    if not os.path.exists("saved_models"):
        os.makedirs("saved_models", exist_ok = True)

    
    # Calculate automated weights from training masks
    train_mask_dir = os.path.join(dataset_dir, "train/masks")
    weights = get_automated_weights(train_mask_dir, num_classes)

    # Dataset Transforms
    img_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    mask_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Lambda(mask_to_binary)
    ])

    val_img_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_mask_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),    
        transforms.ToTensor(),
        transforms.Lambda(mask_to_binary)
    ])


    # Dataloaders
    train_loader, test_loader, val_loader = create_dataloaders(dataset_dir = dataset_dir, batch_size = batch_size, img_transform = img_transform, mask_transform = mask_transform, val_img_transform = val_img_transform, val_mask_transform = val_mask_transform)

    model = UNet(num_classes = num_classes, feature_list = feature_list, bottleneck_size = bottleneck_size).to(device)

    # Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

    loss_function = HybridLoss(
        num_classes = num_classes, 
        weights = weights, 
        alpha=0.5
    ).to(device)

    # Accuracy Metrics
    if num_classes == 1:
        train_acc_metric = BinaryAccuracy().to(device)
        val_acc_metric = BinaryAccuracy().to(device)
    else:
        train_acc_metric = MulticlassAccuracy(num_classes = num_classes).to(device)
        val_acc_metric = MulticlassAccuracy(num_classes = num_classes).to(device)


    best_val_acc = 0.0
    best_epoch = 0
    best_model_state = None

    scheduler = lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min', 
        factor=0.1, 
        patience=10,
        min_lr=1e-6
    )

    # TRAINING AND VALIDATION LOOP

    # Training Loop
    for epoch in range(num_epochs):

        model.train()

        running_train_loss = 0.0

        train_acc_metric.reset()

        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [TRAIN]", leave=False)

        for images, masks in train_pbar:

            images, masks = images.to(device), masks.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = loss_function(outputs, masks)

            loss.backward()

            optimizer.step()

            with torch.no_grad():
                if num_classes == 1:
                    # Logits -> Probabilities -> Threshold
                    preds = (torch.sigmoid(outputs) > 0.5).float()
                    train_acc_metric.update(preds, masks)
                else:
                    # Logits -> Class Index
                    preds = torch.argmax(outputs, dim=1)
                    train_acc_metric.update(preds, masks.squeeze(1))

            running_train_loss += loss.item() * images.size(0)

            train_pbar.set_postfix(batch_loss=loss.item())

        # Average Loss per Sample
        epoch_train_loss = running_train_loss / len(train_loader.dataset)
        epoch_train_acc = train_acc_metric.compute()

        


        # Validation Loop
        model.eval()

        running_val_loss = 0.0
            
        val_acc_metric.reset()

        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [VAL]", leave=False)

        with torch.no_grad():

            for images, masks in val_pbar:

                images, masks = images.to(device), masks.to(device)

                outputs = model(images)

                loss = loss_function(outputs, masks)

                if num_classes == 1:
                    preds = (torch.sigmoid(outputs) > 0.5).float()
                    val_acc_metric.update(preds, masks)

                else:
                    preds = torch.argmax(outputs, dim=1)
                    val_acc_metric.update(preds, masks.squeeze(1))

                running_val_loss += loss.item() * images.size(0)

                val_pbar.set_postfix(batch_loss=loss.item())

        # Average Loss per Sample
        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        epoch_val_acc = val_acc_metric.compute()


        

        scheduler.step(epoch_val_loss)
        
        # Log metrics for the epoch
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")
        
        # --- Check for Best Model ---
        # If current validation accuracy is better than the best seen so far
        if epoch_val_acc > best_val_acc:
            # Update the best validation accuracy
            best_val_acc = epoch_val_acc
            # Store the current epoch number
            best_epoch = epoch + 1
            # Use copy.deepcopy to save a snapshot of the model's state
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save(best_model_state, f"saved_models/best_unet.pth")
            # Print a message indicating a new best model
            print(f"    ^ New best model found!")
        # --- End Check ---

    print("\n--- Training Complete ---")
    
    # --- Load Best Model ---
    # Check if a best model state was saved
    if best_model_state is not None:
        print(f"\nReturning best model from epoch {best_epoch} with {best_val_acc:.4f} validation accuracy.")
        # Load the best performing weights back into the model
        model.load_state_dict(best_model_state)
    else:
        # Warn if no improvement was seen and the last model is being returned
        print("\nWarning: No best model state was saved (e.g., validation never improved). Returning last model.")
    
    # Return the best model
    return model

In [21]:
DATASET_DIR = "data"
LEARNING_RATE = 0.001
NUM_EPOCHS = 1
NUM_CLASSES = 1
FEATURE_LIST = [32, 64, 128, 256, 512]
IMAGE_SIZE = 256
BOTTLENECK_SIZE = 1024
BATCH_SIZE = 8


TrainedUNet = Train_UNet(
    dataset_dir = DATASET_DIR,
    num_epochs = NUM_EPOCHS,
    learning_rate = LEARNING_RATE,
    num_classes = NUM_CLASSES,
    feature_list = FEATURE_LIST,
    bottleneck_size = BOTTLENECK_SIZE,
    image_size = IMAGE_SIZE,
    batch_size = BATCH_SIZE
)

Loading cached weights from weights_cache.pt...


Epoch 1/1 [TRAIN]:   0%|          | 0/1021 [00:00<?, ?it/s]

Epoch 1/1 [VAL]:   0%|          | 0/180 [00:00<?, ?it/s]

Epoch 1/1 | Train Loss: 0.7208, Train Acc: 0.8546 | Val Loss: 0.6142, Val Acc: 0.9300
    ^ New best model found!

--- Training Complete ---

Returning best model from epoch 1 with 0.9300 validation accuracy.
